In [1]:
from utils import *
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = "cpu"
print(f"Using {device} device")

Using mps device


## Linear Regression

Simple population-level linear regression model. The country is not considered.

In [3]:
X, Y, C = load_height_data()
model = LinearRegression().fit(X, Y)
y_pred = model.predict(X)
print("Loss:\n", np.mean((y_pred - Y)**2))
print("RMSD:\n", np.sqrt(np.mean((y_pred - Y)**2)))

Loss:
 45.53098
RMSD:
 6.747665


Next, we try learning a separate linear regression model for each country. Notice the loss is much better.

In [4]:
X, Y, C = load_height_data()
model = ContextLinearRegression().fit(X, Y, C)
y_pred = model.predict(X, C)
print("Loss:\n", np.mean((y_pred - Y)**2))
print("RMSD:\n", np.sqrt(np.mean((y_pred - Y)**2)))

Loss:
 23.753902446756058
RMSD:
 4.87379753854795


## Multilayer Perceptron
The MLP without country as input performs about equally well as learning a separate linear regression model for each country.

In [8]:
model = NeuralNetwork(dim_in=3, dim_out=1, dim_hidden=300, n_hidden=1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = HeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 369.501465  [   50/168000]
loss: 31.514580  [ 5050/168000]
loss: 47.903862  [10050/168000]
loss: 39.276512  [15050/168000]
loss: 42.566490  [20050/168000]
loss: 21.084980  [25050/168000]
loss: 29.106981  [30050/168000]
loss: 22.486507  [35050/168000]
loss: 30.810459  [40050/168000]
loss: 31.797680  [45050/168000]
loss: 27.958963  [50050/168000]
loss: 29.816790  [55050/168000]
loss: 25.977684  [60050/168000]
loss: 29.277449  [65050/168000]
loss: 21.942265  [70050/168000]
loss: 26.116110  [75050/168000]
loss: 23.394178  [80050/168000]
loss: 30.113607  [85050/168000]
loss: 24.991022  [90050/168000]
loss: 25.699638  [95050/168000]
loss: 21.786543  [100050/168000]
loss: 25.972841  [105050/168000]
loss: 21.396269  [110050/168000]
loss: 20.809658  [115050/168000]
loss: 31.596205  [120050/168000]
loss: 32.794106  [125050/168000]
loss: 27.826141  [130050/168000]
loss: 16.638855  [135050/168000]
loss: 21.055878  [140050/168000]
loss: 25.392729  [1450

## Context-Sensitive Network
The task is one-hot-encoded and appended to the input.

In [9]:
model = NeuralNetwork(dim_in=203, dim_out=1, dim_hidden=300, n_hidden=3).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = ContextSensitiveHeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 376.608551  [   50/168000]
loss: 33.479080  [ 5050/168000]
loss: 26.754581  [10050/168000]
loss: 27.605000  [15050/168000]
loss: 25.750448  [20050/168000]
loss: 35.251671  [25050/168000]
loss: 22.347876  [30050/168000]
loss: 24.043642  [35050/168000]
loss: 24.054501  [40050/168000]
loss: 25.936621  [45050/168000]
loss: 28.718672  [50050/168000]
loss: 20.881277  [55050/168000]
loss: 23.985975  [60050/168000]
loss: 24.762445  [65050/168000]
loss: 20.420645  [70050/168000]
loss: 21.690336  [75050/168000]
loss: 18.691792  [80050/168000]
loss: 17.565563  [85050/168000]
loss: 28.833670  [90050/168000]
loss: 27.499870  [95050/168000]
loss: 26.168068  [100050/168000]
loss: 22.573994  [105050/168000]
loss: 24.724680  [110050/168000]
loss: 25.324446  [115050/168000]
loss: 18.576284  [120050/168000]
loss: 22.014198  [125050/168000]
loss: 12.783624  [130050/168000]
loss: 15.698271  [135050/168000]
loss: 21.336439  [140050/168000]
loss: 21.823479  [1450